Support vector machines, the kernel trick, and regularization for support vector machines

In [45]:
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm 
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter
import seaborn as sns
from sklearn.metrics import mean_absolute_error
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.model_selection import cross_val_score
import optuna
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

In [46]:
random_state=42

## IBM

In [47]:
classification_reports = []
classification_report_keys = []

### Data

In [48]:
# Import IBM dataset
df = pd.read_csv('C:/Users/caleb/Projects/BU Spring 2026/Module-B-semester-2/Milestone 3 EDA/ibm_hi_small_trans_cleaned.csv')
df

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Amount Paid,Is Laundering,Amount_Received_USD,Amount_Paid_USD,...,Payment Currency_Yuan,Payment Format_ACH,Payment Format_Bitcoin,Payment Format_Cash,Payment Format_Cheque,Payment Format_Credit Card,Payment Format_Reinvestment,Payment Format_Wire,Account_Same,Bank_Same
0,2022/09/01 00:20,10,8000EBD30,10,8000EBD30,3697.340000,3697.340000,0,3697.340000,3697.340000,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
1,2022/09/01 00:20,3208,8000F4580,1,8000F5340,0.010000,0.010000,0,0.010000,0.010000,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0,0
2,2022/09/01 00:00,3209,8000F4670,3209,8000F4670,14675.570000,14675.570000,0,14675.570000,14675.570000,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
3,2022/09/01 00:02,12,8000F5030,12,8000F5030,2806.970000,2806.970000,0,2806.970000,2806.970000,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
4,2022/09/01 00:06,10,8000F5200,10,8000F5200,36682.970000,36682.970000,0,36682.970000,36682.970000,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5078340,2022/09/10 23:57,54219,8148A6631,256398,8148A8711,0.154978,0.154978,0,3107.386389,3107.386389,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,0
5078341,2022/09/10 23:35,15,8148A8671,256398,8148A8711,0.108128,0.108128,0,2168.020464,2168.020464,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,0
5078342,2022/09/10 23:52,154365,8148A6771,256398,8148A8711,0.004988,0.004988,0,100.011894,100.011894,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,0
5078343,2022/09/10 23:46,256398,8148A6311,256398,8148A8711,0.038417,0.038417,0,770.280058,770.280058,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,1


In [49]:
df.drop(columns=['Timestamp', 'From Bank', 'To Bank', 'Account', 'Account.1'], inplace=True)

### Undersampling

In [50]:
from collections import Counter


X = df.drop(columns='Is Laundering')
y = df['Is Laundering']
print("Original dataset shape:", Counter(y))
rus = RandomUnderSampler(random_state=42)
X_resampled, y_resampled = rus.fit_resample(X, y)

print("Resampled dataset shape:", Counter(y_resampled))

Original dataset shape: Counter({0: 5073168, 1: 5177})
Resampled dataset shape: Counter({0: 5177, 1: 5177})


### Split Data

In [51]:
X_train, X_test, y_train, y_test = train_test_split(X_resampled, 
                                                    y_resampled, 
                                                    test_size=0.2, 
                                                    stratify=y_resampled,  # Maintains class distribution
                                                    random_state=random_state
                                                    )

### Baseline Models
#### (No parameter optimization, feature scaling, or cross validation)

In [52]:
from sklearn.svm import SVC

# Using classifier since this is a fraud dataset with binary Y/N target
model = SVC()

In [53]:
model.fit(X_train, y_train)

,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [54]:
y_pred = model.predict(X_test)

In [55]:
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Baseline SVM Classifier')

              precision    recall  f1-score   support

           0       0.50      1.00      0.67      1036
           1       0.50      0.00      0.00      1035

    accuracy                           0.50      2071
   macro avg       0.50      0.50      0.33      2071
weighted avg       0.50      0.50      0.33      2071



We see poor performance on the positive class, with 0% recall/f1. This could be due to the linear kernal choice or a lack of feature scaling. We can test this assumption during hyper-parameter tuning

### Feature Scaling

In [56]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [57]:
model.fit(X_train_scaled, y_train)

,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [58]:
y_pred = model.predict(X_test_scaled)

In [59]:
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Baseline SVM Classifier')

              precision    recall  f1-score   support

           0       0.88      0.89      0.89      1036
           1       0.89      0.88      0.89      1035

    accuracy                           0.89      2071
   macro avg       0.89      0.89      0.89      2071
weighted avg       0.89      0.89      0.89      2071



Once feature scaling is applied, recall on positive class jumps from 0% to 88%. This shows that feature scaling was critical for the model. We can attempt to further boost performance with hyperparameter tuning

### Parameter Tuning

In [60]:
from sklearn.model_selection import StratifiedKFold

# cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=random_state)
cv = StratifiedKFold(n_splits=5) # Using stratified to maintain class distribution

In [61]:
def objective(trial):
    c = trial.suggest_float("C", 0.001, 1.0)
    kernel = trial.suggest_categorical("kernel", ["rbf", "sigmoid"])
    # degree = trial.suggest_int("degree", 2, 5) # degree of kernal function in case of 'poly'
    coef0 = trial.suggest_float("coef0", 0.0, 1.0) # Independent term in kernel function. It is only significant in ‘poly’ and ‘sigmoid’.
    # shrinking = trial.suggest_categorical("shrinking", [True, False])
    class_weight = trial.suggest_categorical("Class_weight", ['balanced', None])

    model = SVC(
        C=c,
        kernel=kernel,
        # degree=degree,
        coef0=coef0,
        # shrinking=shrinking,
        class_weight=class_weight,
        random_state=random_state,
        cache_size=1000 # FIX 3: Increase RAM cache from default 200MB to 1GB
    )
    
    scores = cross_val_score(
        model, 
        X=X_train_scaled, 
        y=y_train, 
        cv=cv,
        scoring='recall',
        n_jobs=-1,
    )
    recall = scores.mean()
    
    return recall


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

svc_scaled_recall_optuna_results = study.trials_dataframe().sort_values("value", ascending=False)
top_5_svc_scaled_recall_optuna_results = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_svc_scaled_recall_optuna_results)

[I 2026-06-24 20:10:28,737] A new study created in memory with name: no-name-9b45f8a1-4aaf-4083-9dbd-9e52aebb1298
[I 2026-06-24 20:10:31,998] Trial 0 finished with value: 0.8592489641789479 and parameters: {'C': 0.22245793685476278, 'kernel': 'rbf', 'coef0': 0.1253019167260908, 'Class_weight': None}. Best is trial 0 with value: 0.8592489641789479.
[I 2026-06-24 20:10:34,238] Trial 1 finished with value: 0.8597317645961902 and parameters: {'C': 0.5832348431967431, 'kernel': 'rbf', 'coef0': 0.21630519751493105, 'Class_weight': None}. Best is trial 1 with value: 0.8597317645961902.
[I 2026-06-24 20:10:36,424] Trial 2 finished with value: 0.8445210748063847 and parameters: {'C': 0.3667669958645376, 'kernel': 'sigmoid', 'coef0': 0.5568572516190483, 'Class_weight': None}. Best is trial 1 with value: 0.8597317645961902.
[I 2026-06-24 20:10:38,860] Trial 3 finished with value: 0.8597317645961902 and parameters: {'C': 0.43445589741470186, 'kernel': 'rbf', 'coef0': 0.2357354796143274, 'Class_wei

,number,value,datetime_start,datetime_complete,duration,params_C,params_Class_weight,params_coef0,params_kernel,state
54,54,0.943546,2026-06-24 20:11:52.710444,2026-06-24 20:11:55.548319,0 days 00:00:02.837875,0.002061,None,0.374979,sigmoid,COMPLETE
58,58,0.943546,2026-06-24 20:11:59.189440,2026-06-24 20:12:01.868369,0 days 00:00:02.678929,0.001823,None,0.362206,sigmoid,COMPLETE
65,65,0.943546,2026-06-24 20:12:12.198354,2026-06-24 20:12:15.228667,0 days 00:00:03.030313,0.001114,None,0.383836,sigmoid,COMPLETE
63,63,0.943546,2026-06-24 20:12:08.160440,2026-06-24 20:12:10.678552,0 days 00:00:02.518112,0.001958,None,0.448466,sigmoid,COMPLETE
83,83,0.943064,2026-06-24 20:12:47.569306,2026-06-24 20:12:50.676542,0 days 00:00:03.107236,0.003031,None,0.771649,sigmoid,COMPLETE


## Credit Card

In [62]:
classification_reports = []
classification_report_keys = []

### Data

In [63]:
# Import IBM dataset
df = pd.read_csv('C:/Users/caleb/Projects/BU Spring 2026/Module-B-semester-2/Milestone 3 EDA/credit_card_cleaned.csv')
df

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
284802,172786.0,-11.881118,10.071785,-9.834783,-2.066656,-5.364473,-2.606837,-4.918215,7.305334,1.914428,...,0.213454,0.111864,1.014480,-0.509348,1.436807,0.250034,0.943651,0.823731,0.77,0
284803,172787.0,-0.732789,-0.055080,2.035030,-0.738589,0.868229,1.058415,0.024330,0.294869,0.584800,...,0.214205,0.924384,0.012463,-1.016226,-0.606624,-0.395255,0.068472,-0.053527,24.79,0
284804,172788.0,1.919565,-0.301254,-3.249640,-0.557828,2.630515,3.031260,-0.296827,0.708417,0.432454,...,0.232045,0.578229,-0.037501,0.640134,0.265745,-0.087371,0.004455,-0.026561,67.88,0
284805,172788.0,-0.240440,0.530483,0.702510,0.689799,-0.377961,0.623708,-0.686180,0.679145,0.392087,...,0.265245,0.800049,-0.163298,0.123205,-0.569159,0.546668,0.108821,0.104533,10.00,0


### Undersampling

In [64]:
from collections import Counter


X = df.drop(columns='Class')
y = df['Class']
print("Original dataset shape:", Counter(y))
rus = RandomUnderSampler(random_state=42)
X_resampled, y_resampled = rus.fit_resample(X, y)

print("Resampled dataset shape:", Counter(y_resampled))

Original dataset shape: Counter({0: 284315, 1: 492})
Resampled dataset shape: Counter({0: 492, 1: 492})


### Split Data

In [65]:
X_train, X_test, y_train, y_test = train_test_split(X_resampled, 
                                                    y_resampled, 
                                                    test_size=0.2, 
                                                    stratify=y_resampled,  # Maintains class distribution
                                                    random_state=random_state
                                                    )

### Baseline Models
#### (No parameter optimization, feature scaling, or cross validation)

In [66]:
from sklearn.svm import SVC

# Using classifier since this is a fraud dataset with binary Y/N target
model = SVC()

In [67]:
model.fit(X_train, y_train)

,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [68]:
y_pred = model.predict(X_test)

In [69]:
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Baseline SVM Classifier')

              precision    recall  f1-score   support

           0       0.54      0.49      0.52        99
           1       0.53      0.57      0.55        98

    accuracy                           0.53       197
   macro avg       0.53      0.53      0.53       197
weighted avg       0.53      0.53      0.53       197



We see poor performance with macro avg f1 of 0.53 and consistent performance across positive and negative class. This could be due to the linear kernal choice or a lack of feature scaling. We can test this assumption during hyper-parameter tuning

### Feature Scaling

In [70]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [71]:
model.fit(X_train_scaled, y_train)

,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [72]:
y_pred = model.predict(X_test_scaled)

In [73]:
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Baseline SVM Classifier')

              precision    recall  f1-score   support

           0       0.94      0.99      0.97        99
           1       0.99      0.94      0.96        98

    accuracy                           0.96       197
   macro avg       0.97      0.96      0.96       197
weighted avg       0.97      0.96      0.96       197



Once feature scaling is applied, recall on positive class jumps from ~50% to 96%. This shows that feature scaling was critical for the model. We can attempt to further boost performance with hyperparameter tuning

### Parameter Tuning

In [74]:
from sklearn.model_selection import StratifiedKFold

# cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=random_state)
cv = StratifiedKFold(n_splits=5) # Using stratified to maintain class distribution

In [75]:
def objective(trial):
    c = trial.suggest_float("C", 0.001, 1.0)
    kernel = trial.suggest_categorical("kernel", ["rbf", "sigmoid"])
    # degree = trial.suggest_int("degree", 2, 5) # degree of kernal function in case of 'poly'
    coef0 = trial.suggest_float("coef0", 0.0, 1.0) # Independent term in kernel function. It is only significant in ‘poly’ and ‘sigmoid’.
    # shrinking = trial.suggest_categorical("shrinking", [True, False])
    class_weight = trial.suggest_categorical("Class_weight", ['balanced', None])

    model = SVC(
        C=c,
        kernel=kernel,
        # degree=degree,
        coef0=coef0,
        # shrinking=shrinking,
        class_weight=class_weight,
        random_state=random_state,
        cache_size=1000 # FIX 3: Increase RAM cache from default 200MB to 1GB
    )
    
    scores = cross_val_score(
        model, 
        X=X_train_scaled, 
        y=y_train, 
        cv=cv,
        scoring='recall',
        n_jobs=-1,
    )
    recall = scores.mean()
    
    return recall


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

svc_scaled_recall_optuna_results = study.trials_dataframe().sort_values("value", ascending=False)
top_5_svc_scaled_recall_optuna_results = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_svc_scaled_recall_optuna_results)

[I 2026-06-24 20:13:20,751] A new study created in memory with name: no-name-0da5b139-c529-45b2-b2ff-0f62a8480afa
[I 2026-06-24 20:13:20,777] Trial 0 finished with value: 0.8122687439143135 and parameters: {'C': 0.04170726563450069, 'kernel': 'rbf', 'coef0': 0.47488710624729347, 'Class_weight': 'balanced'}. Best is trial 0 with value: 0.8122687439143135.
[I 2026-06-24 20:13:20,793] Trial 1 finished with value: 0.8122038299253489 and parameters: {'C': 0.28985269487989923, 'kernel': 'sigmoid', 'coef0': 0.23741878351181067, 'Class_weight': 'balanced'}. Best is trial 0 with value: 0.8122687439143135.
[I 2026-06-24 20:13:20,808] Trial 2 finished with value: 0.8679974034404415 and parameters: {'C': 0.8413310107526579, 'kernel': 'rbf', 'coef0': 0.2845277791280214, 'Class_weight': None}. Best is trial 2 with value: 0.8679974034404415.
[I 2026-06-24 20:13:20,825] Trial 3 finished with value: 0.8172995780590717 and parameters: {'C': 0.8550742065545504, 'kernel': 'sigmoid', 'coef0': 0.64605223845

,number,value,datetime_start,datetime_complete,duration,params_C,params_Class_weight,params_coef0,params_kernel,state
48,48,0.901266,2026-06-24 20:13:21.763549,2026-06-24 20:13:21.795291,0 days 00:00:00.031742,0.001470,None,0.931381,rbf,COMPLETE
12,12,0.873061,2026-06-24 20:13:20.972415,2026-06-24 20:13:20.987707,0 days 00:00:00.015292,0.985620,None,0.010786,rbf,COMPLETE
16,16,0.873061,2026-06-24 20:13:21.047960,2026-06-24 20:13:21.062949,0 days 00:00:00.014989,0.996775,None,0.379780,rbf,COMPLETE
13,13,0.873061,2026-06-24 20:13:20.987707,2026-06-24 20:13:21.003060,0 days 00:00:00.015353,0.977540,None,0.026491,rbf,COMPLETE
32,32,0.873061,2026-06-24 20:13:21.397461,2026-06-24 20:13:21.427265,0 days 00:00:00.029804,0.929830,None,0.353598,rbf,COMPLETE


In [76]:
top_5_svc_scaled_recall_optuna_results.iloc[0]['params_C']

np.float64(0.0014695513347025013)

In [77]:
model = SVC(
    C=top_5_svc_scaled_recall_optuna_results.iloc[0]['params_C'],
    kernel=top_5_svc_scaled_recall_optuna_results.iloc[0]['params_kernel'],
    # degree=degree,
    coef0=top_5_svc_scaled_recall_optuna_results.iloc[0]['params_coef0'],
    # shrinking=shrinking,
    class_weight=top_5_svc_scaled_recall_optuna_results.iloc[0]['params_Class_weight'],
    random_state=random_state,
    cache_size=1000 # FIX 3: Increase RAM cache from default 200MB to 1GB
)

In [78]:
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
print(classification_report(y_pred=y_pred, y_true=y_test))
None

c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


              precision    recall  f1-score   support

           0       0.00      0.00      0.00        99
           1       0.50      1.00      0.66        98

    accuracy                           0.50       197
   macro avg       0.25      0.50      0.33       197
weighted avg       0.25      0.50      0.33       197



c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Model overfit due to limited number of samples

## PPP

In [79]:
classification_reports = []
classification_report_keys = []

### Data

In [80]:
# Import IBM dataset
df = pd.read_csv('C:/Users/caleb/Projects/BU Spring 2026/Module-B-semester-2/Milestone 3 EDA/ppp_cleaned.csv')
df

,ProcessingMethod,BorrowerState,LoanStatus,Term,InitialApprovalAmount,CurrentApprovalAmount,ServicingLenderState,RuralUrbanIndicator,HubzoneIndicator,LMIIndicator,...,MORTGAGE_INTEREST_PROCEED_pct,RENT_PROCEED_pct,REFINANCE_EIDL_PROCEED_pct,HEALTH_CARE_PROCEED_pct,DEBT_INTEREST_PROCEED_pct,PROCEED_Per_Job,Fraud,DateApproved_int,ForgivenessDate_int,LoanStatusDate_int
0,0.0,48.0,2.0,24,769358.78,769358.78,11.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,12409.01,0,1588291200,1605830400,1608249600
1,0.0,48.0,2.0,24,736927.79,736927.79,11.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,10094.90,0,1588291200,1628726400,1632787200
2,0.0,48.0,2.0,24,691355.00,691355.00,29.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,9218.07,0,1588291200,1612915200,1615939200
3,0.0,48.0,2.0,24,499871.00,499871.00,29.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,23803.38,0,1588291200,1631232000,1634342400
4,0.0,48.0,2.0,24,367437.00,367437.00,37.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,14697.48,0,1588291200,1617840000,1629158400
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
968527,0.0,56.0,2.0,24,150000.00,150000.00,54.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,10000.00,0,1585872000,1607472000,1610496000
968528,0.0,56.0,2.0,24,150000.00,150000.00,31.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,3452.38,0,1586822400,1604361600,1607385600
968529,1.0,56.0,2.0,60,150000.00,150000.00,54.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,29999.40,0,1613088000,1629158400,1631664000
968530,0.0,56.0,2.0,60,150000.00,150000.00,18.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,21428.57,0,1586908800,1645574400,1646697600


### Undersampling

In [81]:
from collections import Counter


X = df.drop(columns='Fraud')
y = df['Fraud']
print("Original dataset shape:", Counter(y))
rus = RandomUnderSampler(random_state=42)
X_resampled, y_resampled = rus.fit_resample(X, y)

print("Resampled dataset shape:", Counter(y_resampled))

Original dataset shape: Counter({0: 968437, 1: 95})
Resampled dataset shape: Counter({0: 95, 1: 95})


### Split Data

In [82]:
X_train, X_test, y_train, y_test = train_test_split(X_resampled, 
                                                    y_resampled, 
                                                    test_size=0.2, 
                                                    stratify=y_resampled,  # Maintains class distribution
                                                    random_state=random_state
                                                    )

### Baseline Models
#### (No parameter optimization, feature scaling, or cross validation)

In [83]:
from sklearn.svm import SVC

# Using classifier since this is a fraud dataset with binary Y/N target
model = SVC()

In [84]:
model.fit(X_train, y_train)

,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [85]:
y_pred = model.predict(X_test)

In [86]:
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Baseline SVM Classifier')

              precision    recall  f1-score   support

           0       1.00      0.74      0.85        19
           1       0.79      1.00      0.88        19

    accuracy                           0.87        38
   macro avg       0.90      0.87      0.87        38
weighted avg       0.90      0.87      0.87        38



We see strong performance already without scaling

### Feature Scaling

In [87]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [88]:
model.fit(X_train_scaled, y_train)

,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [89]:
y_pred = model.predict(X_test_scaled)

In [90]:
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Baseline SVM Classifier')

              precision    recall  f1-score   support

           0       0.93      0.74      0.82        19
           1       0.78      0.95      0.86        19

    accuracy                           0.84        38
   macro avg       0.86      0.84      0.84        38
weighted avg       0.86      0.84      0.84        38



Feature scaling does not affect this dataset

### Parameter Tuning

In [91]:
from sklearn.model_selection import StratifiedKFold

# cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=random_state)
cv = StratifiedKFold(n_splits=5) # Using stratified to maintain class distribution

In [92]:
def objective(trial):
    c = trial.suggest_float("C", 0.001, 1.0)
    kernel = trial.suggest_categorical("kernel", ["rbf", "sigmoid"])
    # degree = trial.suggest_int("degree", 2, 5) # degree of kernal function in case of 'poly'
    coef0 = trial.suggest_float("coef0", 0.0, 1.0) # Independent term in kernel function. It is only significant in ‘poly’ and ‘sigmoid’.
    # shrinking = trial.suggest_categorical("shrinking", [True, False])
    class_weight = trial.suggest_categorical("Class_weight", ['balanced', None])

    model = SVC(
        C=c,
        kernel=kernel,
        # degree=degree,
        coef0=coef0,
        # shrinking=shrinking,
        class_weight=class_weight,
        random_state=random_state,
        cache_size=1000 # FIX 3: Increase RAM cache from default 200MB to 1GB
    )
    
    scores = cross_val_score(
        model, 
        X=X_train_scaled, 
        y=y_train, 
        cv=cv,
        scoring='recall',
        n_jobs=-1,
    )
    recall = scores.mean()
    
    return recall


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

svc_scaled_recall_optuna_results = study.trials_dataframe().sort_values("value", ascending=False)
top_5_svc_scaled_recall_optuna_results = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_svc_scaled_recall_optuna_results)

[I 2026-06-24 20:13:25,386] A new study created in memory with name: no-name-0848c74c-3e96-40b7-92e3-038203284678
[I 2026-06-24 20:13:25,413] Trial 0 finished with value: 0.8816666666666666 and parameters: {'C': 0.103351990102654, 'kernel': 'sigmoid', 'coef0': 0.9616089573954022, 'Class_weight': 'balanced'}. Best is trial 0 with value: 0.8816666666666666.
[I 2026-06-24 20:13:25,427] Trial 1 finished with value: 0.9083333333333332 and parameters: {'C': 0.4194839015078619, 'kernel': 'sigmoid', 'coef0': 0.8293393973371884, 'Class_weight': 'balanced'}. Best is trial 1 with value: 0.9083333333333332.
[I 2026-06-24 20:13:25,442] Trial 2 finished with value: 0.8833333333333332 and parameters: {'C': 0.398787895857029, 'kernel': 'rbf', 'coef0': 0.226571131601629, 'Class_weight': 'balanced'}. Best is trial 1 with value: 0.9083333333333332.
[I 2026-06-24 20:13:25,457] Trial 3 finished with value: 0.8833333333333332 and parameters: {'C': 0.4252435460851366, 'kernel': 'rbf', 'coef0': 0.321546097933

,number,value,datetime_start,datetime_complete,duration,params_C,params_Class_weight,params_coef0,params_kernel,state
1,1,0.908333,2026-06-24 20:13:25.413361,2026-06-24 20:13:25.427660,0 days 00:00:00.014299,0.419484,balanced,0.829339,sigmoid,COMPLETE
10,10,0.908333,2026-06-24 20:13:25.557682,2026-06-24 20:13:25.572842,0 days 00:00:00.015160,0.594755,None,0.688567,sigmoid,COMPLETE
7,7,0.908333,2026-06-24 20:13:25.504133,2026-06-24 20:13:25.519724,0 days 00:00:00.015591,0.775801,balanced,0.132649,sigmoid,COMPLETE
5,5,0.908333,2026-06-24 20:13:25.473971,2026-06-24 20:13:25.488930,0 days 00:00:00.014959,0.814268,None,0.669950,sigmoid,COMPLETE
32,32,0.908333,2026-06-24 20:13:25.935856,2026-06-24 20:13:25.957711,0 days 00:00:00.021855,0.214368,balanced,0.114134,sigmoid,COMPLETE


In [93]:
model = SVC(
    C=top_5_svc_scaled_recall_optuna_results.iloc[0]['params_C'],
    kernel=top_5_svc_scaled_recall_optuna_results.iloc[0]['params_kernel'],
    # degree=degree,
    coef0=top_5_svc_scaled_recall_optuna_results.iloc[0]['params_coef0'],
    # shrinking=shrinking,
    class_weight=top_5_svc_scaled_recall_optuna_results.iloc[0]['params_Class_weight'],
    random_state=random_state,
    cache_size=1000 # FIX 3: Increase RAM cache from default 200MB to 1GB
)

In [94]:
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
print(classification_report(y_pred=y_pred, y_true=y_test))
None

              precision    recall  f1-score   support

           0       1.00      0.74      0.85        19
           1       0.79      1.00      0.88        19

    accuracy                           0.87        38
   macro avg       0.90      0.87      0.87        38
weighted avg       0.90      0.87      0.87        38



The model generalized well